## Modelo XGBoost

XGBoost (*Extreme Gradient Boosting*) es un algoritmo de machine learning supervisado basado en árboles de decisión. Construye un modelo predictivo a través de un proceso iterativo impulsado por la optimización del descenso del gradiente para minimizar la función de pérdida.

En una primera instancia, se entrenó un modelo XGBoost base incluyendo la totalidad de las variables de la temporada 1. Sin embargo, al evaluar su rendimiento sobre el conjunto de datos de la temporada 2, generó predicciones muy deficientes (obteniendo un LogLoss de aproximadamente 2.71).

Al analizar la importancia de las variables, detectamos que el modelo dependía casi exclusivamente de `sz_top` y `sz_bot` (y de las variables que se calculan a partir de ellas) para clasificar los swings.

Para lograr mejores predicciones, en esta notebook vamos a desarrollar y comparar dos modelos:

1. Un modelo excluyendo por completo las variables de la zona de strikes.
2. Un modelo ligeramente superador, implementando *feature engineering* para corregir espacialmente dichas variables.

Finalmente, para garantizar que los modelos alcancen su máximo rendimiento predictivo, implementaremos `FLAML` (Fast and Lightweight AutoML). Esta herramienta nos permitirá realizar una búsqueda automatizada de los hiperparámetros, maximizando la eficiencia computacional y minimizando nuestra métrica de error.

### Carga de librerías

In [4]:
import polars as pl
import pandas as pd
from pathlib import Path
from flaml import AutoML

### Carga de datos de la temporada 1

In [5]:
ROOT = Path("..")
ruta_t1 = ROOT / "data" / "processed" / "temporada1_limpio.parquet"
temporada1 = pl.read_parquet(ruta_t1).to_pandas()

## Primer modelo

### Definición de variables predictoras y respuesta

In [6]:
columnas_excluir = ["pitch_id", "batter", "pitcher", "description", "swing"]
columnas_contaminadas = [
    "sz_top",
    "sz_bot",
    "sz_mid",
    "strike_zone_size",
    "relative_height",
    "pitch_location",
    "is_strike_zone",
    "is_shadow_zone",
    "distance_to_corner",
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(
    columns=[col for col in todas_excluidas if col in temporada1.columns]
)
respuesta = temporada1["swing"]

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=["object", "string"]).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype("category")

print(
    f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras."
)

El dataset tiene 709852 filas y 15 variables predictoras.


### Configuración del modelo XGBoost

La ventaja de usar `FLAML` es que no solo realiza una búsqueda de los hiperparámetros mucho más inteligente que simplemente probar algunas combinaciones al azar, sino que también hace el *K-fold* y las particiones automáticamente.

In [ ]:
# Inicializamos el motor de búsqueda
automl = AutoML()

# Configuramos la búsqueda del mejor modelo
opciones_flaml = {
    "time_budget": 600,  # Tiempo en segundos que le damos a FLAML para buscar
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["xgboost"],  # Optimiza un XGBoost
    "seed": 714,
    "verbose": 0,  # Que no imprima nada en la consola (aunque activándolo tampoco mostraba nada. Quizás es porque estamos trabajando en una nb)
}

automl.fit(X_train=explicativas, y_train=respuesta, **opciones_flaml)

# Resultados finales
print(f"Mejores hiperparámetros encontrados: {automl.best_config}")
print(f"Mejor LogLoss estimado: {automl.best_loss:.4f}")

# Extraemos el modelo ganador entrenado con el conjunto de datos completo de la temporada 1
mejor_modelo = automl.model.estimator

In [ ]:
importancias = mejor_modelo.feature_importances_

df_importancias = pd.DataFrame(
    {"variable": explicativas.columns, "importancia": importancias}
)

df_importancias = df_importancias.sort_values(by="importancia", ascending=False)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10).to_string(index=False))

### Predicción para Kaggle

Primero, leemos los datos limpios de la temporada 2.

In [ ]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio.parquet"

temporada2 = pl.read_parquet(ruta_t2)

Ahora, aplicamos el modelo entrenado con la temporada 1 sobre el modelo de la temporada 2 para realizar las predicciones de swing.

In [ ]:
temporada2 = temporada2.to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2["pitch_id"]

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.drop(
    columns=[col for col in todas_excluidas if col in temporada2.columns]
)

# Pasamos las variables necesarias a categóricas, con las mismas categorías que el training set
columnas_texto = explicativas.select_dtypes(include=["category"]).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(
        X_test[col], categories=explicativas[col].cat.categories
    )

# Reordenamos las columnas para que coincidan con el conjunto de la temporada 1
X_test = X_test[mejor_modelo.feature_names_in_]

# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame(
    {"pitch_id": pitch_ids_t2, "swing_probability": probabilidades_swing}
)

Por último, guardamos el archivo con las predicciones

In [ ]:
ruta_prediccion = ROOT / "outputs" / "xgboost_1" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

## Segundo modelo - Variables de sensor modificadas

In [ ]:
# Leemos nuevamente los datos
temporada1 = pl.read_parquet(ruta_t1).to_pandas()

In [ ]:
# Eliminamos las variables base y las espaciales contaminadas originales
columnas_excluir = ["pitch_id", "batter", "pitcher", "description", "swing"]
columnas_contaminadas = [
    "sz_top_mean",
    "sz_bot_mean",
    "relative_x",  # Sacamos las intermedias para que no introduzcan ruido
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(
    columns=[col for col in todas_excluidas if col in temporada1.columns]
)
respuesta = temporada1["swing"]

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=["object", "string"]).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype("category")

print(
    f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras."
)

In [ ]:
# Inicializamos el motor de búsqueda
automl = AutoML()

# Configuramos la búsqueda del mejor modelo
opciones_flaml = {
    "time_budget": 600,  # Tiempo en segundos que le damos a FLAML para buscar
    "metric": "log_loss",
    "task": "classification",
    "estimator_list": ["xgboost"],  # Optimiza un XGBoost
    "seed": 714,
    "verbose": 0,  # Que no imprima nada en la consola (aunque activándolo tampoco mostraba nada. Quizás es porque estamos trabajando en una nb)
}

automl.fit(X_train=explicativas, y_train=respuesta, **opciones_flaml)

# Extraemos el modelo ganador entrenado con el conjunto de datos completo de la temporada 1
mejor_modelo = automl.model.estimator

# Mostramos los resultados finales
print("-" * 40)
print(f"Mejores hiperparámetros encontrados: {automl.best_config}")
print(f"Mejor LogLoss estimado: {automl.best_loss:.4f}")
print("-" * 40)

### Predicción para Kaggle

Al modelo lo entramos con el promedio de 'sz_top' y 'sz_bot' por bateador. El problema es que el dataset de la temporada 2 puede tener jugadores nuevos para los que no vamos a contar con estas medidas en la temporada 1. Vamos a ver cuántos son estos nuevos jugadores.

In [ ]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio_promedio.parquet"

temporada2 = pl.read_parquet(ruta_t2)

# Guardamos los IDs para el archivo final
pitch_ids_t2 = temporada2["pitch_id"]

# Filtramos X_test
X_test = temporada2.drop(
    columns=[col for col in todas_excluidas if col in temporada2.columns]
)

# Alineamos las variables categóricas con las de entrenamiento
columnas_texto = explicativas.select_dtypes(include=["category"]).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(
        X_test[col], categories=explicativas[col].cat.categories
    )

X_test = X_test[mejor_modelo.feature_names_in_]

# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos y guardamos el DataFrame final
predicciones = pl.DataFrame(
    {"pitch_id": pitch_ids_t2, "swing_probability": probabilidades_swing}
)

ruta_prediccion = ROOT / "outputs" / "xgboost_2" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

Como este modelo dio una predicción sobre el dataset de la temporada 2 ligeramente mejor al primer modelo XGBoost, vamos a deternos a analizar cuáles fueron las principales variables que utilizó para clasificar.

In [ ]:
importancias = mejor_modelo.feature_importances_

df_importancias = pd.DataFrame(
    {"variable": explicativas.columns, "importancia": importancias}
)

df_importancias = df_importancias.sort_values(by="importancia", ascending=False)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10).to_string(index=False))

- `is_strike_zone` (45.69%): la decisión de hacer swing o no depende en gran parte de si la pelota va a la zona válida o no.

- `pfx_x` (12.17%): un lanzamiento con mucho movimiento horizontal engaña al bateador y tiene influencia en la probabilidad de swing.

- `distance_to_corner` (8.28%): el modelo detectó que las pelotas que llegan en las esquinas de la zona de strike generan una duda o un comportamiento distinto en el bateador respecto a las pelotas que van cómodas al centro.

En resumen, la decisión de batear o no depende de si la pelota va a la zona válida (*is_strike_zone_curada*), pero está también muy condicionada por cuánto se mueve la pelota de forma horizontal al llegar a la zona (*pfx_x*) y qué tan cerca del límite pasa (*distance_to_corner*).